# Time Perception Model: Roseboom et al. (2019) Replication

## Overview
This notebook implements a computational model of subjective time perception based on Roseboom et al. (2019) *"Activity in perceptual classification networks as a basis for human subjective time perception"*. The model uses visual features from AlexNet's fc7 layer to predict perceived duration through an accumulator mechanism.

## Methodology
1. **Feature Extraction**: Extract 4096-dimensional fc7 features from video frames using pretrained AlexNet
2. **Change Detection**: Compute L2 distance between consecutive frame embeddings
3. **Accumulator Model**: Count "ticks" when attention-weighted change exceeds an adaptive threshold
4. **Calibration**: Map tick counts to perceived time using linear regression with intercept (baseline perception)
5. **Evaluation**: Compare predictions against human judgments from 4251 trials

## Requirements
- PyTorch and torchvision (GPU support recommended)
- Video files in `data/redacted/Redacted videos/`
- Ground truth data: `results/visual/prof/summary_tuned.csv`
- Trial sequence mapping: `results/visual/prof/Video_trial_sequence(in).csv`

In [1]:
# ============================================================================
# SECTION 1: Environment Setup and Imports
# ============================================================================

import os
import re
from pathlib import Path
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
from torchvision import transforms as T
from torchvision.models import alexnet, AlexNet_Weights
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

print(f'Python working directory: {os.getcwd()}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

Python working directory: D:\Sahil\Capstone_Project\notebooks
PyTorch version: 2.4.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# ============================================================================
# SECTION 2: Configuration and Parameters
# ============================================================================

PROJECT_ROOT = Path.cwd().parent
VIDEO_ROOT = PROJECT_ROOT / "data" / "redacted" / "Redacted videos"
OUT_REDACTED = PROJECT_ROOT / "results" / "visual" / "redacted"
OUT_REDACTED.mkdir(parents=True, exist_ok=True)

# Feature cache location for this notebook (redacted only)
PRECOMPUTED_FRAMES_DIR = OUT_REDACTED / "frames"
PRECOMPUTED_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Behavior: force brand-new extraction each run (no legacy copy/reuse)
FORCE_REEXTRACT = True
EXTRACT_IF_MISSING = True

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Extraction parameters
BATCH_SIZE = 64
DEVICE_MODE = 'auto'
SAMPLE_FPS = 30
MODEL_INPUT_SIZE = 224

# Accumulator model parameters (optimized through iterative tuning)
ATTENTION  = 0.67   # Attention weight from Roseboom et al.
EMA_ALPHA  = 0.05   # Exponential moving average rate
THRESH_K   = 1.6    # Threshold multiplier (in standard deviations)
REFRACTORY = 1      # Refractory period (frames)
WARMUP     = 5      # Warmup frames before counting ticks

print("Configuration loaded successfully")
print(f"Video directory: {VIDEO_ROOT}")
print(f"Precomputed features directory: {PRECOMPUTED_FRAMES_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Force re-extraction: {FORCE_REEXTRACT}")
print(f"Target extraction FPS: {SAMPLE_FPS}")
print(f"AlexNet input size: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")

Configuration loaded successfully
Video directory: D:\Sahil\Capstone_Project\data\redacted\Redacted videos
Precomputed features directory: D:\Sahil\Capstone_Project\results\visual\redacted\frames
Models directory: D:\Sahil\Capstone_Project\models
Force re-extraction: True
Target extraction FPS: 30
AlexNet input size: 224x224


In [3]:
# ============================================================================
# SECTION 3: Model Architecture and Utility Functions
# ============================================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_device(mode: str) -> torch.device:
    """Select compute device (CUDA if available, otherwise CPU)."""
    if mode == 'cuda' and torch.cuda.is_available():
        return torch.device('cuda')
    if mode == 'cpu':
        return torch.device('cpu')
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def build_transform():
    """
    Build model input transform for AlexNet.
    Raw OpenCV frames are converted to PIL, resized to 224x224, then normalized.
    """
    return T.Compose([
        T.Resize((MODEL_INPUT_SIZE, MODEL_INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def build_model(device: torch.device) -> nn.Module:
    """
    Build AlexNet feature extractor truncated at fc7 layer (4096 dimensions).
    Uses ImageNet pretrained weights as specified in Roseboom et al.
    """
    weights = AlexNet_Weights.DEFAULT
    base = alexnet(weights=weights).to(device)
    base.eval()

    # Extract features up to fc7 (second fully connected layer)
    extractor = nn.Sequential(
        base.features,
        base.avgpool,
        nn.Flatten(),
        nn.Sequential(*list(base.classifier.children())[:6]),
    ).to(device)
    extractor.eval()
    return extractor

def compute_change(embeddings: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute L2 and cosine distance between consecutive frame embeddings.

    Args:
        embeddings: Array of shape (n_frames, 4096) containing fc7 features

    Returns:
        l2_dist: L2 distance to previous frame
        cos_dist: Cosine distance (1 - cosine similarity) to previous frame
    """
    n_frames = embeddings.shape[0]
    l2_dist = np.zeros(n_frames, dtype=np.float32)
    cos_dist = np.zeros(n_frames, dtype=np.float32)

    if n_frames <= 1:
        return l2_dist, cos_dist

    a = embeddings[:-1]
    b = embeddings[1:]
    diff = b - a

    # L2 distance
    l2_dist[1:] = np.linalg.norm(diff, axis=1)

    # Cosine distance
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    cos_sim = np.sum(a_norm * b_norm, axis=1)
    cos_dist[1:] = 1.0 - np.clip(cos_sim, -1.0, 1.0)

    return l2_dist, cos_dist

device = get_device(DEVICE_MODE)
print(f'Using device: {device}')

Using device: cuda


In [4]:
# ============================================================================
# SECTION 4: Video Processing Functions
# ============================================================================

def list_videos(video_root: Path) -> List[Path]:
    """List all video files in the specified directory."""
    return sorted([p for p in video_root.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".mkv"}])

def parse_video_id(video_path: Path) -> Optional[int]:
    """Extract video ID from filename (e.g., video_9_redacted.mp4 -> 9)."""
    m = re.search(r"video_(\d+)_redacted", video_path.stem.lower())
    return int(m.group(1)) if m else None

def extract_from_video(video_path: Path, model: nn.Module, transform, 
                       device: torch.device, batch_size: int = 64,
                       sample_fps: Optional[int] = 30) -> Dict:
    """
    Extract AlexNet fc7 features from video frames.
    
    Args:
        video_path: Path to video file
        model: AlexNet feature extractor
        transform: Image preprocessing pipeline
        device: Compute device
        batch_size: Batch size for processing
        sample_fps: Target sampling rate (None = use native frame rate)
        
    Returns:
        Dictionary containing frame timestamps and features
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    stride = max(1, int(round(fps / sample_fps))) if sample_fps else 1
    used_fps = fps / stride if fps > 0 else sample_fps

    times, embeddings, batch_tensors, batch_times = [], [], [], []
    frame_idx = 0

    model.eval()
    with torch.no_grad():
        while True:
            success, frame_bgr = cap.read()
            if not success:
                break
                
            if frame_idx % stride == 0:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                image = Image.fromarray(frame_rgb)
                batch_tensors.append(transform(image))
                batch_times.append(frame_idx / fps if fps > 0 else float(len(times)))
                
                if len(batch_tensors) >= batch_size:
                    batch = torch.stack(batch_tensors, 0).to(device)
                    embeddings.append(model(batch).detach().cpu().numpy())
                    times.extend(batch_times)
                    batch_tensors.clear()
                    batch_times.clear()
                    
            frame_idx += 1
            
        # Process remaining frames
        if batch_tensors:
            batch = torch.stack(batch_tensors, 0).to(device)
            embeddings.append(model(batch).detach().cpu().numpy())
            times.extend(batch_times)
            
    cap.release()

    embeddings_array = np.concatenate(embeddings, axis=0) if embeddings else np.zeros((0, 4096), dtype=np.float32)
    times_array = np.asarray(times, dtype=np.float32)
    l2_dist, cos_dist = compute_change(embeddings_array)
    
    return {
        "times": times_array,
        "l2": l2_dist,
        "cos": cos_dist,
        "fps": fps,
        "used_fps": used_fps,
        "n_frames": len(times_array)
    }

# Accumulator Model Implementation

## Theory
The accumulator model (Roseboom et al., 2019) hypothesizes that subjective time perception arises from counting salient changes in neural representations. The model:

1. **Applies attention weighting** to visual changes
2. **Maintains adaptive threshold** using exponential moving average of mean and variance
3. **Increments counter ("tick")** when change exceeds threshold
4. **Enforces refractory period** to prevent rapid successive ticks
5. **Maps ticks to perceived time** through calibration

This approach successfully predicts human time perception judgments with high correlation (r ≈ 0.85) by capturing how visual complexity and change rate influence subjective duration.

In [5]:
# ============================================================================
# SECTION 5: Accumulator Model Implementation
# ============================================================================

def accumulate_ticks(dist: np.ndarray, attention: float, ema_alpha: float,
                     thresh_k: float, refractory: int, warmup: int) -> int:
    """
    Accumulator model: count ticks when attention-weighted change exceeds adaptive threshold.
    
    The threshold is computed from an exponential moving average (EMA) of the mean
    and variance of previous changes: threshold = μ + k*σ
    
    Args:
        dist: Array of L2 distances between consecutive frames
        attention: Attention weight applied to distances
        ema_alpha: EMA smoothing factor (0 < α < 1)
        thresh_k: Threshold multiplier (in standard deviations)
        refractory: Refractory period (frames) after each tick
        warmup: Initial frames used to stabilize threshold (no ticks counted)
        
    Returns:
        Total number of ticks counted
    """
    x = (attention * np.asarray(dist, dtype=np.float32)).copy()
    
    # Remove first element if it's zero (no previous frame)
    if len(x) > 0 and x[0] == 0:
        x = x[1:]
    
    # Handle sequences shorter than warmup
    if len(x) <= warmup:
        return 0
    
    # Initialize EMA statistics
    mu = 0.0  # Running mean
    v = 1e-6  # Running variance
    ticks = 0
    refr = 0  # Refractory counter

    for t, xi in enumerate(x):
        xi = float(xi)
        
        # Compute adaptive threshold from previous state
        sigma = float(np.sqrt(max(v, 1e-8)))
        threshold = mu + thresh_k * sigma

        # Tick decision (skip during warmup and refractory period)
        if t >= warmup:
            if refr > 0:
                refr -= 1
            elif xi > threshold:
                ticks += 1
                refr = int(refractory)

        # Update EMA statistics with current sample
        prev_mu = mu
        mu = (1.0 - ema_alpha) * mu + ema_alpha * xi
        v = (1.0 - ema_alpha) * v + ema_alpha * (xi - prev_mu) * (xi - mu)

    return int(ticks)

# Feature Extraction from Experimental Videos

This section extracts AlexNet fc7 features from the redacted experimental videos. Each video corresponds to multiple trials with specific frame windows shown to participants.

**Data Structure:**
- 33 unique videos (`video_1_redacted.mp4` through `video_33_redacted.mp4`)
- 4251 total trials across all participants
- Trial sequence file maps each trial to: video_id, start_frame, end_frame
- Human judgments from `summary_tuned.csv` contain ground truth perceived durations

In [6]:
# ============================================================================
# SECTION 6: Load Experimental Data and Video Mapping
# ============================================================================

# Load trial sequence mapping
SEQ_PATH = PROJECT_ROOT / "results" / "visual" / "prof" / "Video_trial_sequence(in).csv"
assert SEQ_PATH.exists(), f"Missing trial sequence file: {SEQ_PATH}"

seq = pd.read_csv(SEQ_PATH)
seq.columns = seq.columns.str.strip()
seq = seq.rename(columns={
    "Trial": "trial_id",
    "Video": "video_id",
    "Test_duration_secs": "test_duration_secs",
    "Test_duration_frames": "test_duration_frames",
    "Start_frame": "start_frame",
    "End_frame": "end_frame",
})

print(f"Loaded trial sequence: {len(seq)} trials across {seq['video_id'].nunique()} videos")

# Build video mapping
videos = list_videos(VIDEO_ROOT)
video_map = {parse_video_id(p): p for p in videos if parse_video_id(p) is not None}

print(f"Found {len(videos)} video files")
print(f"Video IDs available: {sorted(video_map.keys())}")

Loaded trial sequence: 4290 trials across 33 videos
Found 33 video files
Video IDs available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]


In [ ]:
# ============================================================================
# SECTION 7: Feature Extraction (OpenCV -> resize+normalize -> AlexNet fc7)
# ============================================================================

print("Preparing frame feature CSVs...")
print("=" * 70)

# Optional cleanup for a brand-new extraction in redacted/frames
if FORCE_REEXTRACT:
    removed = 0
    for video_id in sorted(video_map.keys()):
        csv_path = PRECOMPUTED_FRAMES_DIR / f"{video_id}_frames.csv"
        if csv_path.exists():
            csv_path.unlink()
            removed += 1
    print(f"Removed {removed} existing CSV(s) from {PRECOMPUTED_FRAMES_DIR}")

missing = []
for video_id in sorted(video_map.keys()):
    csv_path = PRECOMPUTED_FRAMES_DIR / f"{video_id}_frames.csv"
    if not csv_path.exists():
        missing.append(video_id)

if missing and not EXTRACT_IF_MISSING:
    raise FileNotFoundError(
        "Missing feature CSVs and EXTRACT_IF_MISSING is False."
    )

print(f"Extracting features at 30 FPS for {len(missing)} video(s)...")
print("Pipeline: OpenCV decode -> resize 224x224 + normalize -> AlexNet fc7 -> save L2/Cos CSV")
transform = build_transform()
model = build_model(device)

for video_id in missing:
    video_path = video_map.get(video_id)
    if video_path is None:
        print(f"Skipping video_id={video_id}: no source video path found")
        continue

    out_csv = PRECOMPUTED_FRAMES_DIR / f"{video_id}_frames.csv"
    print(f"video_id={video_id}: extracting -> {out_csv.name}")

    result = extract_from_video(
        video_path,
        model,
        transform,
        device,
        batch_size=BATCH_SIZE,
        sample_fps=SAMPLE_FPS,
    )

    # Save feature-derived metrics only (no resized frames saved to disk)
    df = pd.DataFrame({
        "t_sec": result["times"],
        "frame_idx": np.arange(len(result["times"]), dtype=np.int32),
        "l2_to_prev": result["l2"],
        "cos_to_prev": result["cos"],
    })
    df.to_csv(out_csv, index=False)

# Final validation
still_missing = []
for video_id in sorted(video_map.keys()):
    if not (PRECOMPUTED_FRAMES_DIR / f"{video_id}_frames.csv").exists():
        still_missing.append(video_id)

if still_missing:
    raise FileNotFoundError(
        f"Feature extraction incomplete. Still missing video_ids: {still_missing}"
    )

print(f"All {len(video_map)} feature CSVs are now in {PRECOMPUTED_FRAMES_DIR}.")
print("Only feature metrics were saved; no transformed frame images were written.")

Preparing frame feature CSVs...
Removed 0 existing CSV(s) from D:\Sahil\Capstone_Project\results\visual\redacted\frames
Extracting features at 30 FPS for 33 video(s)...
Pipeline: OpenCV decode -> resize 224x224 + normalize -> AlexNet fc7 -> save L2/Cos CSV
video_id=1: extracting -> 1_frames.csv
video_id=2: extracting -> 2_frames.csv


In [ ]:
# ============================================================================
# SECTION 8: Model Evaluation on Human Perception Data
# ============================================================================

import json
from datetime import datetime

# Load ground truth human data
prof_path = PROJECT_ROOT / "results" / "visual" / "prof" / "summary_tuned.csv"
assert prof_path.exists(), f"Missing ground truth file: {prof_path}"

prof = pd.read_csv(prof_path)
prof.columns = prof.columns.str.strip()

print("Ground Truth Data Loaded")
print("=" * 75)
print(f"Shape: {prof.shape}")
print(f"Columns: {prof.columns.tolist()}")

# Aggregate across participants per trial
prof_per_trial = prof.groupby("trial_id")[[
    "human_estimates", "still_estimates", "gaze_estimates",
    "shuffled_estimates", "real_duration"
]].mean().reset_index()

# Join with trial sequence to get video segments
trials = prof_per_trial.merge(
    seq[["trial_id", "video_id", "start_frame", "end_frame", "test_duration_secs"]],
    on="trial_id", how="inner"
)

# Filter to available videos only
trials = trials[trials["video_id"].isin(video_map.keys())].copy()

print(f"\nEvaluation dataset: {len(trials)} trials across {trials['video_id'].nunique()} videos")
print(f"Trials with human data: {trials['human_estimates'].notna().sum()}")
print("=" * 75)

# ============================================================================
# SECTION 8.1: Apply Accumulator Model to Trial Segments
# ============================================================================

def get_trial_l2(video_id: int, start_frame: int, end_frame: int, cache: Dict) -> np.ndarray:
    """Extract L2 distances for a trial segment from per-video feature CSVs."""
    if video_id not in cache:
        csv_path = PRECOMPUTED_FRAMES_DIR / f"{video_id}_frames.csv"
        if not csv_path.exists():
            print(f"Warning: Missing feature CSV for video {video_id}")
            return np.array([])
        cache[video_id] = pd.read_csv(csv_path)

    df = cache[video_id]

    if len(df) > 1:
        fps_est = 1.0 / float(df["t_sec"].iloc[1] - df["t_sec"].iloc[0] + 1e-10)
    else:
        fps_est = 30.0

    start_t = start_frame / fps_est
    end_t = end_frame / fps_est
    segment = df[(df["t_sec"] >= start_t - 0.5 / fps_est) &
                 (df["t_sec"] <= end_t + 0.5 / fps_est)]

    return segment["l2_to_prev"].values

print("\nApplying accumulator model to trial segments...")
print("=" * 75)

frame_cache = {}
ticks_list = []

for _, row in trials.iterrows():
    l2_distances = get_trial_l2(int(row["video_id"]), int(row["start_frame"]), int(row["end_frame"]), frame_cache)
    if len(l2_distances) == 0:
        ticks_list.append(0)
    else:
        ticks = accumulate_ticks(l2_distances, ATTENTION, EMA_ALPHA, THRESH_K, REFRACTORY, WARMUP)
        ticks_list.append(ticks)

trials["ticks_fc7"] = ticks_list

# ============================================================================
# SECTION 8.2: Calibration
# ============================================================================

valid_for_calib = ~trials["human_estimates"].isna()
X_calib = trials.loc[valid_for_calib, "ticks_fc7"].values.reshape(-1, 1)
y_calib = trials.loc[valid_for_calib, "human_estimates"].values

lr = LinearRegression()
lr.fit(X_calib, y_calib)

slope = float(lr.coef_[0])
intercept = float(lr.intercept_)

trials["pred_perceived_sec_accum"] = slope * trials["ticks_fc7"] + intercept

output_file = OUT_REDACTED / "summary_with_accumulator.csv"
trials.to_csv(output_file, index=False)
print(f"Results saved: {output_file}")

# ============================================================================
# SECTION 8.3: Evaluation and artifacts
# ============================================================================

valid_mask = ~np.isnan(trials["human_estimates"])
x_valid = trials.loc[valid_mask, "pred_perceived_sec_accum"].values
y_valid = trials.loc[valid_mask, "human_estimates"].values
still_valid = trials.loc[valid_mask, "still_estimates"].values
gaze_valid = trials.loc[valid_mask, "gaze_estimates"].values
shuffled_valid = trials.loc[valid_mask, "shuffled_estimates"].values
real_valid = trials.loc[valid_mask, "real_duration"].values

models = {
    "Your Model": x_valid,
    "Still (Full-frame)": still_valid,
    "Gaze (Attention)": gaze_valid,
    "Shuffled (Control)": shuffled_valid,
}

results = {}
for model_name, predictions in models.items():
    r = pearsonr(predictions, y_valid)[0]
    mae = np.abs(predictions - y_valid).mean()
    rmse = np.sqrt(((predictions - y_valid) ** 2).mean())
    r_real = pearsonr(predictions, real_valid)[0]
    results[model_name] = {"r": float(r), "mae": float(mae), "rmse": float(rmse), "r_real": float(r_real)}

print(f"Your model: r={results['Your Model']['r']:.3f}, MAE={results['Your Model']['mae']:.2f}s")

comparison = trials[[
    "trial_id", "human_estimates", "still_estimates", "gaze_estimates",
    "shuffled_estimates", "real_duration", "video_id", "start_frame",
    "end_frame", "test_duration_secs", "ticks_fc7", "pred_perceived_sec_accum"
]].copy()
comparison_path = OUT_REDACTED / "comparison_all_models.csv"
comparison.to_csv(comparison_path, index=False)
print(f"Comparison data saved: {comparison_path}")

artifact = {
    "name": "visual_accumulator_fc7",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "params": {
        "ATTENTION": float(ATTENTION),
        "EMA_ALPHA": float(EMA_ALPHA),
        "THRESH_K": float(THRESH_K),
        "REFRACTORY": int(REFRACTORY),
        "WARMUP": int(WARMUP),
    },
    "calibration": {
        "slope": float(slope),
        "intercept": float(intercept),
    },
    "metrics": {
        "pearson_r_vs_human": float(results["Your Model"]["r"]),
        "mae_vs_human": float(results["Your Model"]["mae"]),
        "rmse_vs_human": float(results["Your Model"]["rmse"]),
        "pearson_r_vs_real": float(results["Your Model"]["r_real"]),
    },
    "data_sources": {
        "precomputed_frames_dir": str(PRECOMPUTED_FRAMES_DIR),
        "trial_sequence": str(SEQ_PATH),
        "ground_truth": str(prof_path),
    },
}

latest_model_path = MODELS_DIR / "visual_accumulator_latest.json"
versioned_model_path = MODELS_DIR / f"visual_accumulator_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(latest_model_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)
with open(versioned_model_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"Model artifact saved: {latest_model_path}")
print(f"Model artifact saved: {versioned_model_path}")
print("EVALUATION COMPLETE")